In [ ]:
import numpy as np 
import pandas as pd 
import statsmodels.api as sm

In [ ]:
weather_data = pd.read_csv('/kaggle/input/datasets/micahluftig/dasaxz/other_environmental_variables.csv')
print(weather_data.columns)

In [ ]:


# X = pressure (and maybe day-of-week, season as controls)
# y = daily intake count

X = weather_data[['sea_level_pressure(mb)']]
X = sm.add_constant(X)
y = weather_data['apt_seen']

model = sm.GLM(y, X, family=sm.families.NegativeBinomial()).fit()
print(model.summary())

In [ ]:
import numpy as np
effect = np.exp(-0.0128)
print(f"Multiplier per 1 mb pressure increase: {effect:.4f}")
print(f"% change in intake per 1 mb pressure drop: {(1/effect - 1)*100:.2f}%")

That'll give you something like "each 1 mb drop in pressure is associated with roughly a 1.3% increase in expected daily intake" — which is a genuinely usable, quotable finding, and statistically significant (p < 0.001), consistent with what your original moon-phase project found.
Two things worth being upfront about (in your README and your own judgment):

The dispersion warning matters. That warning — alpha not set, using default alpha=1.0 — means statsmodels didn't actually fit the overdispersion parameter for the negative binomial; it just assumed a fixed value. Given your daily intake data has heavy overdispersion (remember, variance/mean ratio was ~7 in file 1), this default alpha=1.0 is probably wrong, and your std errors/p-values could be off as a result. Fix:

In [ ]:
weather_data['pressure_z'] = (weather_data['sea_level_pressure(mb)'] - weather_data['sea_level_pressure(mb)'].mean()) / weather_data['sea_level_pressure(mb)'].std()

X = weather_data[['pressure_z']]
X = sm.add_constant(X)
y = weather_data['apt_seen']

nb_model = sm.NegativeBinomial(y, X)
nb_result = nb_model.fit(method='bfgs', maxiter=200)
print(nb_result.summary())

This estimates alpha from your actual data rather than assuming it, which is the more defensible version for a portfolio piece.

Pseudo R² of 0.0055 is very low. Pressure is statistically significant, but it explains a tiny fraction of the variation in daily intake — meaning pressure moves the needle, but it's a small factor among many (season, day-of-week, adoption events, etc. probably matter more). That's fine — it doesn't invalidate the app idea — but it does mean the app's honest framing should be "pressure nudges risk up or down by a modest amount," not "pressure predicts your shelter's daily intake." Worth stating that explicitly rather than overselling the effect size.

Once you re-fit with proper alpha estimation, we can take that real percentage and plug it into simulate_year() — the arrival rate becomes a function of the pressure input the user provides, instead of a fixed number. Want to run the corrected NB fit first and see if the coefficient holds up?


This run actually failed — worth catching before you build anything on top of it. A few red flags together:

converged: False
Every std error is nan
Pseudo R² is now negative (-0.1215), which shouldn't really happen and signals the optimizer never found a good fit
Multiple warnings: Hessian inversion failed, divide-by-zero in log, MLE didn't converge

So this fit isn't usable — the coefficients (-0.0125, similar to before) might look consistent with your earlier GLM run, but that's coincidental; you can't trust magnitude or significance from a model that failed to converge.
Likely cause: sea_level_pressure(mb) is on a scale like 990–1030, while your intake counts are small numbers like 20–90. This kind of scale mismatch is a classic cause of numerical instability in MLE optimization — the optimizer struggles when your predictor variable has values in the thousands but coefficients need to be tiny (~0.01) to compensate.
Fix: standardize the pressure variable before fitting

This looks solid now — a real, trustworthy fit:

converged: True ✅
Real standard errors (no more nan) ✅
alpha = 0.1621 — the dispersion parameter got properly estimated this time (not defaulted to 1.0), and it's statistically significant too
Pseudo R² is small but positive (0.0034) — consistent with what you found before: pressure matters, but it's a minor factor among many
The LLR p-value (2.167e-19) tells you the model overall is meaningfully better than nothing, even though R² is small

Converting the standardized coefficient back to something interpretable:
Since you standardized pressure, -0.0765 means "per 1 standard deviation change in pressure," not per 1 mb. To translate it into plain English:

In [ ]:
pressure_std = weather_data['sea_level_pressure(mb)'].std()
print(f"1 SD of pressure = {pressure_std:.2f} mb")

# % change in intake per 1 SD drop in pressure
pct_change_per_sd = (np.exp(-(-0.0765)) - 1) * 100
print(f"% increase in expected intake per 1 SD pressure drop: {pct_change_per_sd:.2f}%")

# % change per 1 mb (dividing the effect by the SD in mb)
pct_change_per_mb = (np.exp(0.0765 / pressure_std) - 1) * 100
print(f"% increase in expected intake per 1 mb pressure drop: {pct_change_per_mb:.3f}%")

That'll give you the clean, quotable version — something like "a 1 mb drop in pressure is associated with roughly a 0.5-1% increase in expected daily intake, holding other factors constant." (exact number depends on what pressure_std comes out to — run it and see).
This is a legitimate, defensible result — small effect size, but statistically real and now numerically sound. You're in good shape to move to the next step: taking this coefficient and building it into simulate_year() so daily arrivals scale with a user-provided pressure/forecast input instead of a fixed rate.
Run that conversion code and share the per-mb percentage — that's the number we'll wire into the simulation next.

So your headline finding is something like: "Each 1 mb drop in sea-level pressure is associated with roughly a 1.3% increase in expected daily shelter intake (p < 0.001), controlling for overdispersion in the data." That's a real, defensible, nicely quantified effect — small in absolute terms, but statistically solid, and exactly what you need to make the app meaningfully weather-responsive rather than just cosmetic.
Now let's wire this into the simulation. The core idea: instead of a fixed negative binomial mean for daily arrivals, the mean now shifts based on the forecasted pressure relative to the historical average.

In [ ]:
# From your regression
import json
with open('/kaggle/input/datasets/micahluftig/monte-carlo-sim-data/simulation_params.json') as f:
    params = json.load(f)

mean_intake = params['mean_intake']

baseline_mean_intake = mean_intake  # from file 1, e.g. 45.42
pct_change_per_mb = 0.01285  # 1.285%, as a decimal

def adjusted_intake_mean(forecast_pressure, historical_avg_pressure, baseline_mean):
    pressure_diff = historical_avg_pressure - forecast_pressure  # positive = pressure drop
    multiplier = (1 + pct_change_per_mb) ** pressure_diff
    return baseline_mean * multiplier

Then in simulate_year(), instead of using fixed n_nb, p_nb the whole time, you'd recompute the negative binomial parameters each day based on that day's (forecasted or simulated) pressure. Since the negative binomial's variance/mean relationship needs to stay consistent, the cleanest way is to keep the dispersion (alpha-equivalent) fixed and just rescale the mean:

In [ ]:
def nb_params_from_mean(mean, dispersion_ratio):
    # dispersion_ratio = variance/mean from your original fit (~7.09)
    var = mean * dispersion_ratio
    p = mean / var
    n = mean * p / (1 - p)
    return n, p

## Summary: Weather Effect on Daily Intake

**Question:** Does atmospheric pressure meaningfully predict daily shelter intake volume?

**Method:** Fit a Negative Binomial regression of daily intake counts on standardized sea-level 
pressure, using MLE with an explicitly estimated dispersion parameter (rather than defaulting 
alpha to 1.0, which understated the data's true overdispersion).

**Result:**
- Pressure is a statistically significant predictor of daily intake (p < 0.001)
- Each 1 mb drop in sea-level pressure is associated with a **~1.29% increase** in expected 
  daily intake, holding other factors constant
- Pseudo R² is small (0.0034), meaning pressure is a real but minor driver — daily intake is 
  influenced by many other factors (day-of-week, season, local events, etc.) not captured here

**Takeaway:** This confirms and quantifies the barometric pressure effect first identified in 
the moon-phase/weather analysis project — pressure drops don't just correlate with intake 
volume, they do so in a statistically robust and now precisely measurable way. This coefficient 
is used downstream to make the Monte Carlo capacity simulation weather-responsive: given a 
short-term pressure forecast, the simulation can estimate how intake risk shifts above or below 
baseline.

**Limitation:** Because pressure only explains a small share of intake variance, this should be 
read as a modest risk adjustment on top of baseline volume, not a standalone predictor of daily 
intake.

In [ ]:
import json

weather_params = {
    "pct_change_per_mb": 0.01285,
    "pressure_std": 5.99,
    "coef_pressure_z": -0.0765,
    "alpha_dispersion": 0.1621,
    "historical_avg_pressure": float(weather_data['sea_level_pressure(mb)'].mean())
}

with open('/kaggle/working/weather_params.json', 'w') as f:
    json.dump(weather_params, f, indent=2)

print(weather_params)